# Fixed-notional CAD basket sensitivity
Independent CAD1m intervals, four equal starting weights, all nine fixed models, no compounding or annualization. Costs are total interval budgets, not estimated execution costs. Calendar-year splits are descriptive.

In [ ]:
from pathlib import Path
import hashlib, json
import pandas as pd
ROOT = Path.cwd().resolve()
if not (ROOT / "reports").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "config/evaluation.json").exists(), "Run from repository root or notebooks/"
def checked_csv(folder, name):
    path = ROOT / "reports" / folder
    manifest = json.loads((path / "manifest.json").read_text())
    assert manifest["status"] == "passed"
    assert hashlib.sha256((path / name).read_bytes()).hexdigest() == manifest["artifacts"][name]
    return pd.read_csv(path / name)


In [ ]:
summary = checked_csv("phase6", "summary.csv")
base = summary.loc[summary.period.eq("all") & summary.cost_bps.eq(0)]
print(base[["model", "confidence", "scored_rows", "mean_var_cad", "mean_es_cad"]].to_string(index=False))

In [ ]:
index = ["period", "model", "confidence"]
gross = summary.loc[summary.cost_bps.eq(0)].set_index(index)
costly = summary.loc[summary.cost_bps.eq(10)].set_index(index)
import numpy as np
np.testing.assert_allclose(costly.mean_es_cad - gross.mean_es_cad, 1000, atol=1e-6)
np.testing.assert_array_equal(costly.exceedances, gross.exceedances)
print("CAD1,000 cost shift verified; exceedance counts unchanged.")

In [ ]:
import matplotlib.pyplot as plt
view = base.loc[base.confidence.eq(.99)]
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(view.model, view.mean_es_cad)
ax.set_xlabel("Mean gross 99% ES (CAD) on independent CAD1m intervals")
fig.tight_layout()
plt.show()

Run `make portfolio` to rebuild the sensitivity from the verified Phase 5 snapshot. Nonlinear monetary ES is calculated from the entire transformed tail; applying the return-to-loss function to log-return ES is incorrect. USD spot series are excluded from this CAD book.